### **Historical Data - Succesful adaptation calculator**

In [5]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold

# 1. Load the data
df = pd.read_csv('CompleteDataset.csv')
df['Latitude_country_player'] = df['Latitude_country_player'].astype(str).str.replace(',', '.').astype(float)
df['Longitude_country_player'] = df['Longitude_country_player'].astype(str).str.replace(',', '.').astype(float)
df['club_country_latitude'] = df['club_country_latitude'].astype(str).str.replace(',', '.').astype(float)
df['club_country_longitude'] = df['club_country_longitude'].astype(str).str.replace(',', '.').astype(float)

# 2. Filter the transfers    
df = df.sort_values(['player_name', 'end_year', 'max_play_date'])
df['prev_club_name'] = df['club_name'].shift(1)
df['prev_minutes_ratio'] = df['completed_minutes_ratio'].shift(1)
df['prev_season'] = df['season'].shift(1)

df['transferred'] = (
    (df['club_name'] != df['prev_club_name']) & df['prev_club_name'].notna()
).astype(int)

df = df[df['transferred'] == 1].reset_index(drop=True)

# 3. Define realistic min and max values for normalization
metric_ranges = {
    'goals': (0, 30),
    'conceeded_goals': (0, 80),
    'assists': (0, 20),
    'tot_clean_sheets': (0, 20),
    'defense_score': (0, 100),
    'interceptions_score': (0, 100),
    'sprint_score': (0, 100),
    'crossing_score': (0, 100),
    'short_passing_score': (0, 100),
    'vision_score': (0, 100),
    'dribbling_score': (0, 100),
    'finishing_score': (0, 100)
}

# 4. Normalization function
def normalize_metric(value, min_val, max_val):
    return max(0, min(1, (value - min_val) / (max_val - min_val))) * 100

# 5. Adaptation score function
def calculate_adaptation_score(player):
    prev_ratio = player.get('prev_minutes_ratio', np.nan)
    current_ratio = player.get('completed_minutes_ratio', np.nan)
    
    if pd.notna(prev_ratio) and prev_ratio > 0:
        participation_score = (current_ratio / prev_ratio) * 100
    else:
        participation_score = 0

    performance_metrics = {}
    for metric, (min_val, max_val) in metric_ranges.items():
        value = player.get(metric, np.nan)
        if pd.notna(value):
            performance_metrics[metric] = normalize_metric(value, min_val, max_val)
        else:
            performance_metrics[metric] = 0

    position = player.get('field_sub_position', '')
    performance_score = 0

    if position == 'Goalkeeper':
        performance_score = performance_metrics['tot_clean_sheets'] - performance_metrics['conceeded_goals'] * 0.25
    elif position == 'Centre-Back':
        performance_score = performance_metrics['defense_score'] * 0.5 + performance_metrics['interceptions_score'] * 0.5
    elif position in ['Right-Back', 'Left-Back']:
        performance_score = performance_metrics['sprint_score'] * 0.5 + performance_metrics['crossing_score'] * 0.5
    elif position in ['Defensive Midfield', 'Central Midfield']:
        performance_score = performance_metrics['short_passing_score'] * 0.5 + performance_metrics['interceptions_score'] * 0.5
    elif position in ['Attacking Midfield', 'Left Midfield', 'Right Midfield']:
        performance_score = performance_metrics['vision_score'] * 0.4 + performance_metrics['assists'] * 0.6
    elif position in ['Left Winger', 'Right Winger']:
        performance_score = (performance_metrics['dribbling_score'] + performance_metrics['goals'] + performance_metrics['assists']) / 3
    elif position in ['Centre-Forward', 'Second Striker']:
        performance_score = performance_metrics['finishing_score'] * 0.5 + performance_metrics['goals'] * 0.5

    normalized_participation = max(0, min(1, participation_score / 100))
    normalized_performance = max(0, min(1, performance_score / 100))
    
    weight_performance = 0.6
    weight_participation = 0.4
    adaptation_score = (normalized_participation*weight_participation) + (normalized_performance*weight_performance)

    return normalized_participation, normalized_performance, adaptation_score

# 6. Apply the adaptation score function
df_transfer = df.copy()
df_transfer[['participation_score', 'performance_score', 'adaptation_score']] = df_transfer.apply(
    lambda row: pd.Series(calculate_adaptation_score(row)), axis=1
)

# 7. Binary variable for successful adaptation using fixed 70% threshold
df_transfer['successful_adaptation'] = (df_transfer['adaptation_score'] >= 0.7).astype(int)

# 8. Save to CSV
df_transfer.to_csv('SuccessfulAdaptationHistoricData.csv', index=False)

# 9. Evaluate different weight combinations
weights = [(0.0, 1.0), (0.1, 0.9), (0.2, 0.8), (0.3, 0.7), (0.4, 0.6), (0.5, 0.5), (0.6, 0.4), (0.7, 0.3), (0.8, 0.2), (0.9, 0.1), (1.0, 0.0)]
results = []

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for p_weight, perf_weight in weights:
    df_temp = df_transfer.copy()

    df_temp['adaptation_score_custom'] = (
        df_temp['participation_score'] * p_weight +
        df_temp['performance_score'] * perf_weight
    )

    df_temp['successful_adaptation_custom'] = (df_temp['adaptation_score_custom'] >= 0.7).astype(int)

    if df_temp['successful_adaptation_custom'].nunique() < 2:
        continue  # Avoid single-class scenarios

    df_temp = df_temp.drop(columns=['player_name', 'season', 'country','club_country','club_name', 'field_position', 'field_sub_position', 'prev_club_name', 
                      'prev_season', 'transferred','pref_foot', 'adaptation_score','player_id','prev_club_name','prev_season','end_year','max_play_date'])

    X = df_temp.drop(columns=['successful_adaptation_custom', 'adaptation_score_custom', 'participation_score', 'performance_score'])
    X = X.fillna(0)
    y = df_temp['successful_adaptation_custom']
    
    for train_index, test_index in skf.split(X, y):
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)
        

        # Logistic Regression model
        model = LogisticRegression()
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]

        acc = accuracy_score(y_test, y_pred)
        auc = roc_auc_score(y_test, y_proba)

    #Calculate the average for the adaptation score
    normalized_participation = df_temp['participation_score'].mean()
    normalized_performance = df_temp['performance_score'].mean()
    adapation_score_avg = df_temp['adaptation_score_custom'].mean()
    

    results.append({
        'participation_weight': p_weight,
        'performance_weight': perf_weight,
        'accuracy': round(acc, 4),
        'roc_auc': round(auc, 4),
        'adaptation_score_avg': round(adapation_score_avg, 4),
        'normalized_participation': round(normalized_participation, 4),
        'normalized_performance': round(normalized_performance, 4)
    })

# 10. Display sorted results
results_df = pd.DataFrame(results).sort_values(by='roc_auc', ascending=False)
print("\nResults for each combination of weights:")
print(results_df)
# 11. Save the results to a CSV file
results_df.to_csv('AdaptationScoreResults.csv', index=False)



Results for each combination of weights:
    participation_weight  performance_weight  accuracy  roc_auc  \
4                    0.4                 0.6    1.0000   1.0000   
10                   1.0                 0.0    0.9887   0.9997   
9                    0.9                 0.1    0.9876   0.9996   
8                    0.8                 0.2    0.9851   0.9992   
7                    0.7                 0.3    0.9784   0.9983   
3                    0.3                 0.7    0.9701   0.9939   
5                    0.5                 0.5    0.9569   0.9889   
6                    0.6                 0.4    0.9536   0.9878   
2                    0.2                 0.8    0.9425   0.9845   
1                    0.1                 0.9    0.9207   0.9701   
0                    0.0                 1.0    0.8983   0.9566   

    adaptation_score_avg  normalized_participation  normalized_performance  
4                 0.5948                     0.719                   0.512  